# **Título:** *Análise Socioeconômica da Baixada Fluminense via Ciência de Dados: Um Estudo sobre Precarização do Trabalho e Distribuição de Renda*

## 1. Listar e Navegar na Pasta do FTP

### PNAD Contínua (Microdados - IBGE)

In [ ]:
from ftplib import FTP
import zipfile
import os

print("Conectando ao servidor FTP do IBGE...")
ftp = FTP('ftp.mtps.gov.br' if False else 'ftp.ibge.gov.br')
ftp.login() # Login anônimo automático

# Caminho da pasta de microdados trimestrais de 2024
caminho_ftp = '/Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_continua/Trimestral/Microdados/2024'
ftp.cwd(caminho_ftp)

# Lista os arquivos reais presentes na pasta
arquivos = ftp.nlst()
print("Arquivos encontrados no servidor:", arquivos)

# Seleciona o arquivo do 1º trimestre (ou o primeiro zip da lista)
arquivo_alvo = [arq for arq in arquivos if arq.endswith('.zip')][0]
print(f"Iniciando download do arquivo: {arquivo_alvo}...")

# Faz o download direto via FTP
with open(arquivo_alvo, 'wb') as f:
    ftp.retrbinary(f'RETR {arquivo_alvo}', f.write)

ftp.quit()
print("Download concluído com sucesso!")

# Descompacta o arquivo baixado
pasta_destino = './pnad_2024_trim1'
with zipfile.ZipFile(arquivo_alvo, 'r') as z:
    z.extractall(pasta_destino)

print(f"Arquivos extraídos na pasta '{pasta_destino}':", os.listdir(pasta_destino))

Conectando ao servidor FTP do IBGE...
Arquivos encontrados no servidor: ['PNADC_012024_20250815.zip', 'PNADC_022024_20260324.zip', 'PNADC_032024_20250815.zip', 'PNADC_042024_20250815.zip']
Iniciando download do arquivo: PNADC_012024_20250815.zip...
Download concluído com sucesso!
Arquivos extraídos na pasta './pnad_2024_trim1': ['PNADC_012024.txt']


### CAGED (Ministério do Trabalho e Emprego)

In [ ]:
from ftplib import FTP

# Conecta ao servidor FTP do MTE (login anônimo automático)
ftp = FTP('ftp.mtps.gov.br')
ftp.login()

# Navega até a pasta de interesse (ex: Novo CAGED de 2024)
ftp.cwd('/pdet/microdados/NOVO CAGED/2024/')

# Lista os arquivos disponíveis
arquivos = ftp.nlst()
print("Arquivos encontrados no servidor:")
for arq in arquivos[:5]:
    print("-", arq)

ftp.quit()

Arquivos encontrados no servidor:
- 202401
- 202402
- 202403
- 202404
- 202405


'221 Goodbye.'

### 2. Baixar o Arquivo Diretamente

####CAGED

In [ ]:
from ftplib import FTP
import os

# 1. Conecta ao servidor do Ministério do Trabalho
ftp = FTP('ftp.mtps.gov.br')
ftp.login()
ftp.set_pasv(True)  # Ativa modo passivo para evitar bloqueios no Colab

# 2. Navega até a pasta de 2024
caminho_caged_base = '/pdet/microdados/NOVO CAGED/2024'
ftp.cwd(caminho_caged_base)

# Obtém a lista de subdiretórios mensais
meses = ftp.nlst()
print(f"Pastas de meses encontradas: {meses}")

# Escolhe o primeiro mês disponível (ou um mês específico, ex: '202401')
# Você pode ajustar isso para escolher um mês diferente, se necessário
if meses:
    mes_selecionado = meses[0]
    caminho_caged_completo = os.path.join(caminho_caged_base, mes_selecionado)
    print(f"Navegando para: {caminho_caged_completo}")
    ftp.cwd(caminho_caged_completo)

    # 3. Mapeia os arquivos contidos na pasta do mês
    arquivos = ftp.nlst()
    print(f"Arquivos encontrados em {caminho_caged_completo}: {arquivos}")

    # 4. Filtra e seleciona o primeiro arquivo de movimentação (CAGEDMOV) no formato .7z
    arquivo_alvo = [arq for arq in arquivos if 'CAGEDMOV' in arq and arq.endswith('.7z')]

    if arquivo_alvo:
        arquivo_para_baixar = arquivo_alvo[0]
        print(f"Baixando o arquivo: {arquivo_para_baixar}...")

        # 5. Faz o download binário do arquivo selecionado
        with open(arquivo_para_baixar, 'wb') as local_file:
            ftp.retrbinary(f'RETR {arquivo_para_baixar}', local_file.write)

        print("Download do Novo CAGED concluído com sucesso!")
    else:
        print(f"Nenhum arquivo CAGEDMOV .7z encontrado em {caminho_caged_completo}.")
else:
    print(f"Nenhum diretório mensal encontrado em {caminho_caged_base}.")

ftp.quit()


Pastas de meses encontradas: ['202401', '202402', '202403', '202404', '202405', '202406', '202407', '202408', '202409', '202410', '202411', '202412']
Navegando para: /pdet/microdados/NOVO CAGED/2024/202401
Arquivos encontrados em /pdet/microdados/NOVO CAGED/2024/202401: ['CAGEDEXC202401.7z', 'CAGEDFOR202401.7z', 'CAGEDMOV202401.7z']
Baixando o arquivo: CAGEDMOV202401.7z...
Download do Novo CAGED concluído com sucesso!


'221 Goodbye.'

### 2.1 Leitura e Filtragem em Chunks (PNAD)

In [ ]:
import os
import pandas as pd

# 1. Defina a pasta onde você descompactou os arquivos
pasta_extraida = "./pnad_2024_trim1"  # Ajuste para a pasta criada no unzip

# 2. Localiza automaticamente o arquivo .txt dentro da pasta
arquivos_txt = [os.path.join(pasta_extraida, f) for f in os.listdir(pasta_extraida) if f.endswith('.txt')]

if not arquivos_txt:
    raise FileNotFoundError(f"Nenhum arquivo .txt foi encontrado dentro de '{pasta_extraida}'. Verifique se a etapa de unzip foi executada.")

caminho_txt = arquivos_txt[0]
print(f"Lendo o arquivo localizado em: {caminho_txt}")

# 3. Processamento e filtragem
registros_rj_metrop = []

with open(caminho_txt, 'r', encoding='latin1') as f:
    for linha in f:
        uf = linha[4:6]          # Posições 5-6 (base 0)
        rm_ride = linha[13:15]    # Posições 14-15

        # Filtra Estado do RJ (33) e Região Metropolitana do Rio (17)
        if uf == '33' and rm_ride == '17':
            v2007 = linha[97:98]     # Sexo
            v2009 = linha[98:101]    # Idade
            v4029 = linha[182:183]   # Carteira assinada
            vd4019 = linha[215:223]  # Rendimento habitual

            registros_rj_metrop.append({
                'UF': uf,
                'RM_RIDE': rm_ride,
                'Sexo': v2007.strip(),
                'Idade': v2009.strip(),
                'Carteira_Assinada': v4029.strip(),
                'Rendimento': vd4019.strip()
            })

df_pnad_rio = pd.DataFrame(registros_rj_metrop)

print(f"Total de entrevistas filtradas na RM do Rio: {len(df_pnad_rio)}")
print(df_pnad_rio.head())

Lendo o arquivo localizado em: ./pnad_2024_trim1/PNADC_012024.txt
Total de entrevistas filtradas na RM do Rio: 0
Empty DataFrame
Columns: []
Index: []
